## Task 1. 白盒模型水印

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from tqdm import tqdm

%load_ext autoreload
%autoreload 2
from Week14_utils_Question import LeNet5, evaluate_dataloader, compute_match_rate, random_index_generator

In [ ]:
loss_func = torch.nn.CrossEntropyLoss()
learning_rate = 0.1
epoches = 10
batch_size = 128
LAMBDA = 1.
T = 64
embedding_size = 80 * 10

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST(root="./MNIST", train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root="./MNIST", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=True, drop_last=True)

#### 训练与水印嵌入

In [ ]:
target_b = torch.ones(T)
# 创建一个800行、64列的矩阵X_wm，极度稀疏（大部分为0），每列只有一个1、一个-1
X_wm = np.zeros((embedding_size, T))
rand_idx_gen = random_index_generator(embedding_size)
for col in range(T):
    X_wm[next(rand_idx_gen)][col] = 1.
    X_wm[next(rand_idx_gen)][col] = -1.
X_wm = torch.Tensor(X_wm)

lenet5 = LeNet5()
optimizer = torch.optim.SGD(lenet5.parameters(), lr=learning_rate)

for epoch in range(1, epoches + 1):
    for img, label in tqdm(train_loader):
        # TODO: 完成白盒水印的嵌入训练
        # NOTE: 你可以使用lenet5.get_weight()函数，获取模型最后一层的白盒权重矩阵
        # NOTE: 可能用到的API: torch.sigmoid, torch.nn.functional.binary_cross_entropy

        weight = 
        loss = 

        loss.backward()
        optimizer.step()

    evaluate_dataloader(lenet5, test_loader)
    mr = compute_match_rate(lenet5, X_wm, target_b)  # TODO: 记得完成Week14_utils_Question.py中的compute_match_rate()函数
    print(f"[Epoch {epoch}] Watermark Match Rate: {mr:.4f}")

In [ ]:
final_mr = compute_match_rate(lenet5, X_wm, target_b)
print("Final Watermark Match Rate:", final_mr)

In [ ]:
# 将模型与水印参数保存为一个字典
checkpoint = {
    'model_state_dict': lenet5.state_dict(),
    'X_wm': X_wm,
    'target_b': target_b
}
torch.save(checkpoint, "white_watermark_checkpoint.pth")
print("Model saved.")